# Импорт библиотек и настройка среды

In [1]:
import pandas as pd

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_colwidth', None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Удобства для навигации

In [3]:
navigation = dict()

# Загрузка датасета

Загрузка датасета.

In [10]:
data = pd.read_csv("../data/cs-training.csv")
data = data.drop(columns=["Unnamed: 0"])

In [11]:
navigation["переменная датасета"] = "data"

Улучшенное наименование столбцов.

In [12]:
data = data.rename(columns={
    "SeriousDlqin2yrs": "target",
    "RevolvingUtilizationOfUnsecuredLines": "revolving_utilization",
    "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "num_30_59_days_late",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "num_open_credit_lines",
    "NumberOfTimes90DaysLate": "num_90_days_late",
    "NumberRealEstateLoansOrLines": "num_real_estate_loans",
    "NumberOfTime60-89DaysPastDueNotWorse": "num_60_89_days_late",
    "NumberOfDependents": "num_dependents"
})

# Краткая характеристика датасета

Словарь данных.

In [17]:
data_dictionary = pd.DataFrame({
    "column": data.columns,
    "description": [
        "Наличие у клиента просрочки 90 и более дней или более серьёзное нарушение платёжной дисциплины.",
        "Общий баланс по кредитным картам и личным кредитным линиям, кроме недвижимости и долгов в рассрочку, например автокредитов, делённый на сумму кредитных лимитов.",
        "Возраст заёмщика в годах.",
        "Количество раз, когда заёмщик имел просрочку 30–59 дней, но не хуже, за последние 2 года.",
        "Ежемесячные выплаты по долгам, алименты и расходы на проживание, делённые на ежемесячный валовый доход.",
        "Ежемесячный доход.",
        "Количество открытых кредитов, например автокредит или ипотека, и кредитных линий, например кредитных карт.",
        "Количество раз, когда заёмщик имел просрочку 90 дней или более.",
        "Количество ипотечных и других кредитов, связанных с недвижимостью, включая кредитные линии под залог жилья.",
        "Количество раз, когда заёмщик имел просрочку 60–89 дней, но не хуже, за последние 2 года.",
        "Количество иждивенцев в семье, не включая самого заёмщика: супруг/супруга, дети и т. д."
    ],
    "types": [
        "binary",
        "percent",
        "integer",
        "count",
        "percent",
        "float",
        "count",
        "count",
        "count",
        "count",
        "count"
    ],
    "role": [
        "target",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature"
    ]
})

In [18]:
navigation["словарь признаков"] = "data_dictionary"

In [19]:
data_dictionary

,column,description,types,role
0,target,Наличие у клиента просрочки 90 и более дней или более серьёзное нарушение платёжной дисциплины.,binary,target
1,revolving_utilization,"Общий баланс по кредитным картам и личным кредитным линиям, кроме недвижимости и долгов в рассрочку, например автокредитов, делённый на сумму кредитных лимитов.",percent,feature
2,age,Возраст заёмщика в годах.,integer,feature
3,num_30_59_days_late,"Количество раз, когда заёмщик имел просрочку 30–59 дней, но не хуже, за последние 2 года.",count,feature
4,debt_ratio,"Ежемесячные выплаты по долгам, алименты и расходы на проживание, делённые на ежемесячный валовый доход.",percent,feature
5,monthly_income,Ежемесячный доход.,float,feature
6,num_open_credit_lines,"Количество открытых кредитов, например автокредит или ипотека, и кредитных линий, например кредитных карт.",count,feature
7,num_90_days_late,"Количество раз, когда заёмщик имел просрочку 90 дней или более.",count,feature
8,num_real_estate_loans,"Количество ипотечных и других кредитов, связанных с недвижимостью, включая кредитные линии под залог жилья.",count,feature
9,num_60_89_days_late,"Количество раз, когда заёмщик имел просрочку 60–89 дней, но не хуже, за последние 2 года.",count,feature


**Временная схема:**

Важно отметить, какую временную схему имеют данные. Изобразить её можно примерно так:

<div align="center">
    <img src="../../docs/images/time_scheme.png" width="700">
</div>

То есть скоринг происходит в момент $T_{0}$, признаки относятся к прошлому (период никак не ограничен, кроме как в признаках `num_30_59_days_late` и `num_60_89_days_late`, данные столбцы отражают информацию не ранее чем за 2 года до $T_{0}$). Целевая переменная (`target`) относится к будущему, следующие 2 года после $T_{0}$.

Таким образом, явных признаков temporal leakage не обнаружено.

Объём портфеля

In [20]:
data.shape[0]

150000

In [21]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   target                 150000 non-null  int64  
 1   revolving_utilization  150000 non-null  float64
 2   age                    150000 non-null  int64  
 3   num_30_59_days_late    150000 non-null  int64  
 4   debt_ratio             150000 non-null  float64
 5   monthly_income         120269 non-null  float64
 6   num_open_credit_lines  150000 non-null  int64  
 7   num_90_days_late       150000 non-null  int64  
 8   num_real_estate_loans  150000 non-null  int64  
 9   num_60_89_days_late    150000 non-null  int64  
 10  num_dependents         146076 non-null  float64
dtypes: float64(4), int64(7)
memory usage: 12.6 MB


Как данные выглядят.

In [22]:
data.sample(5)

,target,revolving_utilization,age,num_30_59_days_late,debt_ratio,monthly_income,num_open_credit_lines,num_90_days_late,num_real_estate_loans,num_60_89_days_late,num_dependents
11463,0,0.0114,75,0,0.0081,"6,666.0000",5,0,0,0,0.0000
96070,0,0.0843,63,0,"1,041.0000",NaN,3,0,1,0,0.0000
32480,0,0.0144,65,0,0.3167,"4,000.0000",23,0,1,0,2.0000
133076,0,0.0237,68,0,0.1200,"13,964.0000",16,0,0,0,2.0000
27659,0,0.0222,59,0,"2,093.0000",NaN,8,0,1,0,NaN


Наличие пропусков.

In [23]:
data.isna().sum()

target                       0
revolving_utilization        0
age                          0
num_30_59_days_late          0
debt_ratio                   0
monthly_income           29731
num_open_credit_lines        0
num_90_days_late             0
num_real_estate_loans        0
num_60_89_days_late          0
num_dependents            3924
dtype: int64

Наличие дубликатов.

In [24]:
data.duplicated().sum()

np.int64(609)

Общий bad rate.

In [26]:
round(data["target"].mean() * 100, 4)

np.float64(6.684)

**Вывод по блоку:**
1. Предварительно датасет насчитывает порядка 150 тысяч заёмщиков (есть дубликаты);
2. Каждый заёмщик характеризуется 10 признаками;
3. По описанию данных признаки рассматриваются как информация, доступная на момент $T_{0}$, а `target` относится к следующим двум годам. Поэтому явных признаков temporal leakage не обнаружено;
4. В данных присутствуют пропуски в признаках `monthly_income` и `num_dependents`, их нужно будет тщательно проанализировать;
5. bad rate по всему набору данных составляет порядка 6.684%. Классы сильно несбалансированы, нужно это будет учитывать.

# Data quality

## Возможные риски, связанные с схемой данных

**Сразу обсужу проблемы и риски связанные с самой схемой данных, на которые нужно будет делать поправки при анализе:**

1. `revolving_utilization`:

    Поскольку признак является отношением задолженности к кредитному лимиту, его экстремальные значения могут возникать как из-за действительно высокой утилизации, так и вследствие особенностей расчёта при очень малом/нулевом лимите или ошибок данных. Исходная документация не описывает обработку таких случаев, поэтому экстремальные значения требуют отдельной проверки
2. `debt_ratio`:

    1. Во-первых, данный признак имеет аналогичную проблему с делением на ноль и возможность получить неадекватное значение (так как данный признак является отношением).
    2. Во-вторых, `debt_ratio` агрегирует несколько видов обязательных платежей, поэтому одинаковое значение показателя может соответствовать разной экономической структуре расходов. Из-за отсутствия отдельных компонент детальная интерпретация признака ограничена.

Данные потенциальные проблемы сильно снижают предсказуемость значений данных признаков (особенно `debt_ratio`), что может существенно сказать на качестве анализа. **Поэтому, ответственным за схему данных и данные признаки рекомендуется:**
1. Сформировать чёткие соглашения, что будет делаться в случае, когда знаменатель не известен или равен нулю;
2. Разделить признак `debt_ratio` по смыслу на несколько признаков (например, на `debt_rate`, `alimony_rate` и `rent_rate`) для повышения интерпретируемости.